# Experiment F: Representation Search for 10-Class Clustering

Test multiple data representations to find which one produces the cleanest
10 clusters corresponding to UR, UL, OR, OL, USR, USL, OSR, OSL, FB, BF.

Individual representations + combinations, all evaluated on the same data.

In [ ]:
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
!pip install umap-learn -q
import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')

In [ ]:
import glob, re, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score, f1_score
from collections import Counter, defaultdict
import umap
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
# ── Functions (same cycle extraction as V11 v2) ──────────────
CONFIG = {
    'FS': 50.0, 'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21, 'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0, 'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0, 'TARGET_LEN': 64, 'MIN_CYCLE_SAMPLES': 10,
}
KNOWN_PATTERNS = {'overhand','sneak_overhand','underhand','sneak_underhand','dragon_roll','underhand_default'}
LR_CLASSES = {'UR','UL','OR','OL','USR','USL','OSR','OSL','FB','BF'}
COARSE_MAP = {'UR':'underhand','UL':'underhand','U_generic':'underhand',
    'OR':'overhand','OL':'overhand','O_generic':'overhand',
    'USR':'sneak_underhand','USL':'sneak_underhand','SU_generic':'sneak_underhand',
    'OSR':'sneak_overhand','OSL':'sneak_overhand','SO_generic':'sneak_overhand',
    'FB':'dragon_roll','BF':'dragon_roll','DR_generic':'dragon_roll'}
FINE_MAP = {'UR':'UR','ur':'UR','UR0':'UR','UR-CW':'UR','UL0':'UL',
    'OR':'OR','or':'OR','OR2':'OR','OR3':'OR','OR-OSL':'OR','OR/OSL?':'OR',
    'OL':'OL','ol':'OL','OL2':'OL','USR':'USR','USL':'USL','OSR':'OSR','OSL':'OSL',
    'FB':'FB','fb':'FB','FB2':'FB','BF2':'BF','bf':'BF',
    'underhand':'U_generic','overhand':'O_generic','dragon_roll':'DR_generic',
    'sneak_underhand':'SU_generic','sneak_overhand':'SO_generic',
    'CW':None,'cw':None,'CCW':None,'ccw':None,'idle':None,'Idle3':None,'Idle-or-ol?':None,'VQ5':None,'vq16':None}
_FP = [(re.compile(r'^USR$',re.I),'USR'),(re.compile(r'^USL$',re.I),'USL'),
    (re.compile(r'^OSR$',re.I),'OSR'),(re.compile(r'^OSL$',re.I),'OSL'),
    (re.compile(r'^UR',re.I),'UR'),(re.compile(r'^UL',re.I),'UL'),
    (re.compile(r'^OR',re.I),'OR'),(re.compile(r'^OL',re.I),'OL'),
    (re.compile(r'^FB',re.I),'FB'),(re.compile(r'^BF',re.I),'BF'),
    (re.compile(r'^CW$',re.I),None),(re.compile(r'^CCW$',re.I),None),
    (re.compile(r'^idle',re.I),None),(re.compile(r'^vq',re.I),None)]
def map_fine(raw):
    raw = raw.strip()
    if raw in FINE_MAP: return FINE_MAP[raw]
    if raw.lower() in FINE_MAP: return FINE_MAP[raw.lower()]
    for p,c in _FP:
        if p.match(raw): return c
    return None
def extract_signals(df):
    t=df['timestamp_ms'].values/1000.0; A=df[['ax_w','ay_w','az_w']].values
    om=df[['gx','gy','gz']].values*(np.pi/180.0); return t,A,om
def detect_cycles(t,om,fs=50.0):
    md=np.linalg.norm(om,axis=1)*(180/np.pi); n=len(md)
    if n<7: return []
    w=int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if w%2==0: w+=1
    w=max(5,min(w,n if n%2==1 else n-1)); po=max(1,min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']),w-2))
    ms=savgol_filter(md,window_length=w,polyorder=po,mode='interp')
    ms=savgol_filter(ms,window_length=w,polyorder=po,mode='interp')
    pks,_=find_peaks(ms,distance=max(1,int(CONFIG['CYCLE_MIN_PERIOD_S']*fs)),
        prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(pks)==0: return []
    cyc=[]
    for i,p in enumerate(pks):
        l=0 if i==0 else (pks[i-1]+p)//2; r=(len(t)-1) if i==len(pks)-1 else (p+pks[i+1])//2
        if r>l and (r-l)>=CONFIG['MIN_CYCLE_SAMPLES']: cyc.append((l,r))
    return cyc
def pair_cycles(t0,c0,t1,c1):
    p0,p1,u=[],[],set()
    for a in c0:
        bi,bo=-1,-1.0
        for i,b in enumerate(c1):
            if i in u: continue
            o=max(0,min(t0[a[1]],t1[b[1]])-max(t0[a[0]],t1[b[0]]))
            if o>bo: bo,bi=o,i
        if bi>=0 and bo>0: p0.append(a);p1.append(c1[bi]);u.add(bi)
    return p0,p1
def resample(s,n):
    if len(s)<2: return np.zeros(n) if s.ndim==1 else np.zeros((n,s.shape[1]))
    xo,xn=np.linspace(0,1,len(s)),np.linspace(0,1,n)
    if s.ndim==1: return np.interp(xn,xo,s)
    return np.column_stack([np.interp(xn,xo,s[:,j]) for j in range(s.shape[1])])
def build_cm(A0,A1,om0,om1,s0,e0,s1,e1):
    tl=CONFIG['TARGET_LEN']
    r0=resample(np.column_stack([A0[s0:e0],om0[s0:e0]]),tl)
    r1=resample(np.column_stack([A1[s1:e1],om1[s1:e1]]),tl)
    return np.column_stack([r0,r1]).T.astype(np.float32)
print('Functions defined')

In [ ]:
# ── Load all cycles, keep raw cycle matrices (12,64) ─────────
all_cms = []; all_fine = []; all_groups = []
def load_c(d0p,d1p,name,fine='unlabeled',windows=None):
    df0,df1=pd.read_csv(d0p),pd.read_csv(d1p)
    t0,A0,om0=extract_signals(df0); t1,A1,om1=extract_signals(df1)
    c0=detect_cycles(t0,om0,CONFIG['FS']); c1=detect_cycles(t1,om1,CONFIG['FS'])
    p0,p1=pair_cycles(t0,c0,t1,c1)
    if windows:
        fp0,fp1=[],[]
        for (s0,e0),(s1,e1) in zip(p0,p1):
            if any(ws<=(t0[s0]+t0[e0])/2<we for ws,we in windows): fp0.append((s0,e0));fp1.append((s1,e1))
        p0,p1=fp0,fp1
    g=name.split('/')[0] if '/' in name else name; cnt=0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        cm=build_cm(A0,A1,om0,om1,s0,e0,s1,e1)
        all_cms.append(cm); all_fine.append(fine); all_groups.append(g); cnt+=1
    return cnt
# Homogeneous
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED,'*_device0_processed.csv'))):
    d1=d0.replace('_device0_','_device1_')
    if not os.path.exists(d1): continue
    stem=os.path.basename(d0).replace('_device0_processed.csv','')
    parts=stem.split('_')
    if len(parts)<3: continue
    rest='_'.join(parts[2:]); fine='unlabeled'
    for pat in sorted(KNOWN_PATTERNS,key=len,reverse=True):
        if rest.startswith(pat):
            if pat in ('underhand','underhand_default'): fine='U_generic'
            elif pat=='overhand': fine='O_generic'
            elif pat=='dragon_roll': fine='DR_generic'
            elif pat=='sneak_underhand': fine='SU_generic'
            elif pat=='sneak_overhand': fine='SO_generic'
            break
    load_c(d0,d1,stem,fine)
# Heterogeneous
if os.path.isdir(NEW_LABELED_RAW):
    for sn in sorted(os.listdir(NEW_LABELED_RAW)):
        sd=os.path.join(NEW_LABELED_RAW,sn)
        if not os.path.isdir(sd): continue
        lp=None
        for fn in ('labels_corrected.json','labels.json'):
            p=os.path.join(sd,fn)
            if os.path.isfile(p): lp=p; break
        if not lp: continue
        d0=os.path.join(DATA_PROCESSED,sn+'_device0_processed.csv')
        d1=os.path.join(DATA_PROCESSED,sn+'_device1_processed.csv')
        if not(os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lp,encoding='utf-8') as f: data=_json.load(f)
        segs=data.get('segments',data) if isinstance(data,dict) else data
        wbl=defaultdict(list)
        for seg in segs:
            fi=map_fine(seg.get('label',''))
            if fi is None: continue
            s,e=seg.get('start'),seg.get('end')
            if s is None: continue
            if e is None: e=s+2.0
            wbl[fi].append((float(s),float(e)))
        for fi,wins in sorted(wbl.items()): load_c(d0,d1,sn+'/'+fi,fi,wins)

CMS = np.array(all_cms)  # (N, 12, 64)
y = np.array(all_fine)
lr_mask = np.array([l in LR_CLASSES for l in y])
y_lr = y[lr_mask]
print(f'Total: {len(CMS)} cycles, L/R labeled: {lr_mask.sum()}')
print(f'Distribution: {dict(Counter(y_lr))}')

In [ ]:
# ── Build all representations ─────────────────────────────────
fs = CONFIG['FS']

def eval_repr(X, name, k=10):
    """StandardScale, PCA(50), K-means(k), evaluate NMI/sil/purity on L/R labels."""
    sc = StandardScaler(); Xs = sc.fit_transform(X)
    nc = min(50, Xs.shape[1], Xs.shape[0]-1)
    if nc < Xs.shape[1]:
        pca = PCA(n_components=nc); Xp = pca.fit_transform(Xs)
    else:
        Xp = Xs
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    cl = km.fit_predict(Xp)
    sil = silhouette_score(Xp, cl) if len(set(cl)) > 1 else 0
    cl_lr = cl[lr_mask]
    nmi = normalized_mutual_info_score(y_lr, cl_lr)
    ari = adjusted_rand_score(y_lr, cl_lr)
    # Purity
    correct = sum(max(Counter(y_lr[cl_lr==c]).values(), default=0) for c in range(k))
    purity = correct / len(y_lr)
    # LOSO nearest-centroid F1 on L/R labeled only
    X_lr_s = Xs[lr_mask]; X_lr_p = Xp[lr_mask]; g_lr = np.array(all_groups)[lr_mask]
    ugroups = np.unique(g_lr)
    cg = defaultdict(set)
    for l,g in zip(y_lr,g_lr): cg[l].add(g)
    sing = {c for c,gs in cg.items() if len(gs)<2}
    tgroups = [g for g in ugroups if not set(y_lr[g_lr==g]).issubset(sing)]
    at,ap = [],[]
    for g in tgroups:
        te=g_lr==g; tr=~te
        sc2=StandardScaler(); Xtr=sc2.fit_transform(X[lr_mask][tr]); Xte=sc2.transform(X[lr_mask][te])
        nc2=min(50,Xtr.shape[1],Xtr.shape[0]-1)
        if nc2<Xtr.shape[1]:
            pca2=PCA(n_components=nc2); Xtr=pca2.fit_transform(Xtr); Xte=pca2.transform(Xte)
        km2=KMeans(n_clusters=k,random_state=42,n_init=20,max_iter=500)
        km2.fit(Xtr)
        ytr=y_lr[tr]
        cmap={c:Counter(ytr[km2.labels_==c]).most_common(1)[0][0] if (km2.labels_==c).sum()>0 else 'unk' for c in range(k)}
        preds=[cmap.get(c,'unk') for c in km2.predict(Xte)]
        yte=y_lr[te]
        valid=np.array([p!='unk' for p in preds])
        at.extend(yte[valid].tolist()); ap.extend(np.array(preds)[valid].tolist())
    f1 = f1_score(at, ap, average='macro', zero_division=0) if at else 0
    print(f'  {name:<45s} dim={X.shape[1]:>5d}  NMI={nmi:.3f}  ARI={ari:.3f}  sil={sil:.3f}  pur={purity:.3f}  LOSO-F1={f1:.3f}')
    return {'name':name,'dim':X.shape[1],'nmi':nmi,'ari':ari,'sil':sil,'purity':purity,'f1':f1}

results = []
print('Evaluating representations (k=10):')
print(f'  {"Representation":<45s} {"dim":>5s}  {"NMI":>5s}  {"ARI":>5s}  {"sil":>5s}  {"pur":>5s}  {"LOSO":>7s}')

In [ ]:
# ── R1: Raw 768D (baseline) ───────────────────────────────────
X_raw = CMS.reshape(len(CMS), -1)
results.append(eval_repr(X_raw, 'R1: Raw 768D'))

# ── R2: Amplitude-normalized ───────────────────────────────────
norms = np.linalg.norm(X_raw, axis=1, keepdims=True) + 1e-8
X_ampnorm = X_raw / norms
results.append(eval_repr(X_ampnorm, 'R2: Amplitude-normalized 768D'))

# ── R3: Per-channel z-score per cycle ─────────────────────────
X_chz = np.zeros_like(X_raw)
for i in range(len(CMS)):
    cm = CMS[i]  # (12, 64)
    for ch in range(12):
        mu = cm[ch].mean(); std = cm[ch].std() + 1e-8
        X_chz[i, ch*64:(ch+1)*64] = (cm[ch] - mu) / std
results.append(eval_repr(X_chz, 'R3: Per-channel z-scored 768D'))

In [ ]:
# ── R4: FFT per channel (magnitude + phase) ──────────────────
fft_feats = []
for cm in CMS:
    feats = []
    for ch in range(12):
        F = np.fft.rfft(cm[ch])  # complex
        feats.extend(np.abs(F).tolist())  # magnitude
        feats.extend(np.angle(F).tolist())  # phase (preserves sign/direction)
    fft_feats.append(feats)
X_fft = np.array(fft_feats, dtype=np.float32)
results.append(eval_repr(X_fft, 'R4: FFT mag+phase per channel'))

# ── R5: FFT magnitude only ─────────────────────────────────────
fft_mag = []
for cm in CMS:
    feats = []
    for ch in range(12):
        feats.extend(np.abs(np.fft.rfft(cm[ch])).tolist())
    fft_mag.append(feats)
X_fft_mag = np.array(fft_mag, dtype=np.float32)
results.append(eval_repr(X_fft_mag, 'R5: FFT magnitude only'))

In [ ]:
# ── R6: Signed axis features (18D) ────────────────────────────
# Mean signed gx, gy, gz per hand + std per axis
signed_feats = []
for cm in CMS:
    # D0: channels 3,4,5 = gx,gy,gz; D1: channels 9,10,11
    feats = []
    for ch in [3,4,5,9,10,11]:  # signed gyro channels
        feats.append(np.mean(cm[ch]))  # signed mean
        feats.append(np.std(cm[ch]))   # variability
        feats.append(np.mean(np.sign(cm[ch])))  # sign dominance (-1 to +1)
    signed_feats.append(feats)
X_signed = np.array(signed_feats, dtype=np.float32)
results.append(eval_repr(X_signed, 'R6: Signed axis features (18D)'))

# ── R7: Extended signed features (36D) ─────────────────────────
ext_signed = []
for cm in CMS:
    feats = []
    for ch in [3,4,5,9,10,11]:
        sig = cm[ch]
        feats.extend([np.mean(sig), np.std(sig), np.mean(np.sign(sig)),
                      np.max(sig), np.min(sig),
                      np.sum(np.diff(np.sign(sig))!=0)/len(sig)])  # zero-crossing rate
    ext_signed.append(feats)
X_ext_signed = np.array(ext_signed, dtype=np.float32)
results.append(eval_repr(X_ext_signed, 'R7: Extended signed (36D)'))

In [ ]:
# ── R8: Phase portrait descriptors ─────────────────────────────
# Plot omega_x vs omega_y vs omega_z for each hand = 3D curve
# Compute shape features of the curve
phase_feats = []
for cm in CMS:
    feats = []
    for hand_start in [3, 9]:  # D0 gyro, D1 gyro
        wx = cm[hand_start]; wy = cm[hand_start+1]; wz = cm[hand_start+2]
        # Cross-axis correlations (signed)
        n = len(wx)
        cxy = np.corrcoef(wx, wy)[0,1] if n > 2 else 0
        cxz = np.corrcoef(wx, wz)[0,1] if n > 2 else 0
        cyz = np.corrcoef(wy, wz)[0,1] if n > 2 else 0
        # Phase lags (signed)
        def phase_lag(a, b):
            ac = a - a.mean(); bc = b - b.mean()
            corr = np.correlate(ac, bc, mode='full')
            return (np.argmax(corr) - (len(a)-1)) / fs
        pxy = phase_lag(wx, wy); pxz = phase_lag(wx, wz); pyz = phase_lag(wy, wz)
        # Swept area (cross product magnitude)
        w = np.column_stack([wx, wy, wz])
        cross = np.cross(w[:-1], w[1:])
        area = np.mean(np.linalg.norm(cross, axis=1)) if len(cross) > 0 else 0
        # Axis energy ratios
        ex = np.mean(wx**2); ey = np.mean(wy**2); ez = np.mean(wz**2)
        etot = ex + ey + ez + 1e-8
        feats.extend([cxy, cxz, cyz, pxy, pxz, pyz, area, ex/etot, ey/etot, ez/etot])
    phase_feats.append(feats)
X_phase = np.array(phase_feats, dtype=np.float32)
results.append(eval_repr(X_phase, 'R8: Phase portrait descriptors (20D)'))

In [ ]:
# ── R9: 3D trajectory shape from Fourier reconstruction ──────
K_HARM = 10; LAM = 1e-4
def fourier_recon_shape(accel_3d, fs=50.0, K=10, lam=1e-4):
    N = len(accel_3d)
    if N < 4: return np.zeros(3*(2*K+1))
    T = N/fs; omega = 2*np.pi/T; t = np.arange(N)/fs
    nc = 2*K+1; Phi = np.zeros((N,nc)); W = np.zeros(nc)
    for k in range(1,K+1):
        kw = k*omega
        Phi[:,2*k-1] = -(kw**2)*np.sin(kw*t)
        Phi[:,2*k] = -(kw**2)*np.cos(kw*t)
        W[2*k-1] = kw**3; W[2*k] = kw**3
    coeffs = []
    for ax in range(3):
        a = accel_3d[:,ax]
        A = Phi.T@Phi + lam*np.diag(W**2); b = Phi.T@a
        try: c = np.linalg.solve(A,b)
        except: c = np.zeros(nc)
        c[0] = 0; coeffs.append(c)
    return np.concatenate(coeffs)

traj_feats = []
for cm in CMS:
    acc_d0 = cm[0:3].T  # (64,3)
    acc_d1 = cm[6:9].T
    fc0 = fourier_recon_shape(acc_d0, fs, K_HARM, LAM)
    fc1 = fourier_recon_shape(acc_d1, fs, K_HARM, LAM)
    traj_feats.append(np.concatenate([fc0, fc1]))
X_traj = np.array(traj_feats, dtype=np.float32)
X_traj = np.nan_to_num(X_traj, nan=0, posinf=0, neginf=0)
results.append(eval_repr(X_traj, 'R9: Fourier trajectory coefficients'))

In [ ]:
# ── COMBINATIONS ─────────────────────────────────────────────

# C1: Amplitude-norm + signed axis
X_c1 = np.hstack([X_ampnorm, X_signed])
results.append(eval_repr(X_c1, 'C1: AmpNorm + Signed'))

# C2: Amplitude-norm + phase portrait
X_c2 = np.hstack([X_ampnorm, X_phase])
results.append(eval_repr(X_c2, 'C2: AmpNorm + Phase'))

# C3: Per-channel z-score + signed axis
X_c3 = np.hstack([X_chz, X_signed])
results.append(eval_repr(X_c3, 'C3: ChZ + Signed'))

# C4: Per-channel z-score + phase portrait
X_c4 = np.hstack([X_chz, X_phase])
results.append(eval_repr(X_c4, 'C4: ChZ + Phase'))

# C5: FFT mag+phase + signed axis
X_c5 = np.hstack([X_fft, X_signed])
results.append(eval_repr(X_c5, 'C5: FFT + Signed'))

# C6: Signed + Phase + Trajectory
X_c6 = np.hstack([X_signed, X_phase, X_traj])
results.append(eval_repr(X_c6, 'C6: Signed + Phase + Trajectory'))

# C7: AmpNorm + Signed + Phase + Trajectory (everything physics)
X_c7 = np.hstack([X_ampnorm, X_signed, X_phase, X_traj])
results.append(eval_repr(X_c7, 'C7: AmpNorm + Signed + Phase + Traj'))

# C8: ChZ + FFT + Signed + Phase (everything)
X_c8 = np.hstack([X_chz, X_fft, X_signed, X_phase])
results.append(eval_repr(X_c8, 'C8: ChZ + FFT + Signed + Phase'))

# C9: AmpNorm + FFT mag only
X_c9 = np.hstack([X_ampnorm, X_fft_mag])
results.append(eval_repr(X_c9, 'C9: AmpNorm + FFT mag'))

In [ ]:
# ── SUMMARY TABLE ─────────────────────────────────────────────
print('='*100)
print('EXPERIMENT F: REPRESENTATION SEARCH SUMMARY')
print('='*100)

# Sort by LOSO F1
results_sorted = sorted(results, key=lambda r: r['f1'], reverse=True)

print(f'{"Rank":>4s}  {"Representation":<45s} {"dim":>5s}  {"NMI":>5s}  {"ARI":>5s}  {"sil":>6s}  {"Purity":>6s}  {"LOSO-F1":>7s}')
print('-'*95)
for i, r in enumerate(results_sorted):
    print(f'{i+1:>4d}  {r["name"]:<45s} {r["dim"]:>5d}  {r["nmi"]:.3f}  {r["ari"]:.3f}  {r["sil"]:>6.3f}  {r["purity"]:>6.3f}  {r["f1"]:>7.3f}')

print(f'\nBest by LOSO F1: {results_sorted[0]["name"]} (F1={results_sorted[0]["f1"]:.3f})')
best_nmi = max(results, key=lambda r: r['nmi'])
print(f'Best by NMI:     {best_nmi["name"]} (NMI={best_nmi["nmi"]:.3f})')
best_pur = max(results, key=lambda r: r['purity'])
print(f'Best by purity:  {best_pur["name"]} (purity={best_pur["purity"]:.3f})')
print(f'\nReference: E2 nearest-centroid k=15 F1=0.432, V08 supervised F1=0.632')

# Bar chart
fig, ax = plt.subplots(figsize=(14, 6))
names = [r['name'].split(': ')[1] if ': ' in r['name'] else r['name'] for r in results_sorted]
f1s = [r['f1'] for r in results_sorted]
colors = ['#5DCAA5' if r['f1'] == max(f1s) else '#7F77DD' for r in results_sorted]
bars = ax.barh(range(len(names)), f1s, color=colors)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel('LOSO F1 (10-class nearest centroid)')
ax.set_title('Representation Search: which encoding clusters best into 10 L/R classes?')
ax.axvline(0.432, color='red', ls='--', lw=1, label='E2 baseline (0.432)')
ax.legend(); plt.tight_layout(); plt.show()